In [8]:
# install dependencies if not installed
%pip install opendatasets
%pip install scikit-learn
%pip install legacy-cgi
%pip install pandas

Note: you may need to restart the kernel to use updated packages.


In [9]:
import opendatasets as od
# download data set from kaggle
UCI_Poker = 'https://www.kaggle.com/datasets/rasvob/uci-poker-hand-dataset/data?select=poker-hand-testing.data'
od.download(UCI_Poker)

Skipping, found downloaded files in ".\uci-poker-hand-dataset" (use force=True to force download)


In [10]:
import os
# ensure dataset downloaded
os.listdir('./uci-poker-hand-dataset')

['poker-hand-testing.data', 'poker-hand-training-true.data']

In [11]:
# process data
import pandas as pd
from sklearn.svm import SVC
from sklearn.model_selection import GridSearchCV

data_train = pd.read_csv(filepath_or_buffer="./uci-poker-hand-dataset/poker-hand-training-true.data", sep=',', header=None)
data_test = pd.read_csv(filepath_or_buffer="./uci-poker-hand-dataset/poker-hand-testing.data", sep=',', header=None)

X_train = data_train.values[:,0:10] #all rows of first 10 cols
y_train = data_train.values[:,10] #all rows of 11th col

X_test = data_test.values[:,0:10]
y_test = data_test.values[:,10]

In [12]:
# use scaler on data for SVM, extremely sensitive to feature scale unlike other models we are using
from sklearn.preprocessing import StandardScaler

scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)

In [13]:
# Hyperparameter tuning
params = {
    'C': [0.1, 1, 10, 100],          # regularization strength
    'kernel': ['rbf', 'poly'],        # rbf is usually best for non-linear data
    'gamma': ['scale', 'auto'],       # kernel coefficient
}

model = SVC(decision_function_shape='ovr')  # one-vs-rest for multiclass

print("Starting tuning")
grid = GridSearchCV(model, params, cv=5, scoring="accuracy", n_jobs=-1, verbose=1)
grid.fit(X_train_scaled, y_train)

print("Best set of hyperparameters: ", grid.best_params_)
print("Best score: ", grid.best_score_)

Best set of hyperparameters:  {'C': 10, 'gamma': 'scale', 'kernel': 'rbf'}
Best score:  0.5580167932826869


In [14]:
# metrics
from sklearn.metrics import accuracy_score, f1_score, roc_auc_score, confusion_matrix, classification_report

train_preds = grid.predict(X_train_scaled)
test_preds = grid.predict(X_test_scaled)

accuracy_train = accuracy_score(y_train, train_preds)
accuracy_test = accuracy_score(y_test, test_preds)
print(f"Train Accuracy: {accuracy_train:.2f}")
print(f"Test Accuracy: {accuracy_test:.2f}")
print()

f1_train = f1_score(y_train, train_preds, average="weighted")
f1_test = f1_score(y_test, test_preds, average="weighted")
print(f"F1 Score (train): {f1_train:.2f}")
print(f"F1 Score (test): {f1_test:.2f}")

# Confusion matrix and classification report
print("Confusion Matrix (Test):")
print(confusion_matrix(y_test, test_preds))
print("\nClassification Report (Test):")
print(classification_report(y_test, test_preds, digits=3, zero_division=0))

Train Accuracy: 0.65
Test Accuracy: 0.56

F1 Score (train): 0.62
F1 Score (test): 0.53
Confusion Matrix (Test):
[[366961 134214      4      0      2     26      0      0      0      2]
 [229660 192719     65      6      5     16      0      1      5     21]
 [ 19705  27869     35      9      2      0      1      0      0      1]
 [  5749  15341     25      4      2      0      0      0      0      0]
 [    82   3798      1      0      1      0      0      0      0      3]
 [  1324    225      0      0      0    435      0      0      4      8]
 [   299   1114      9      2      0      0      0      0      0      0]
 [    16    209      3      2      0      0      0      0      0      0]
 [     1      6      0      0      0      5      0      0      0      0]
 [     0      2      0      0      0      0      0      0      0      1]]

Classification Report (Test):
              precision    recall  f1-score   support

           0      0.588     0.732     0.652    501209
           1     